# Extrinsic Downstream Task Evaluation
## Sinhala Script Robustness — Unicode vs Romanized

This notebook evaluates instruct-tuned LLMs on three downstream tasks to measure how romanized (phonetically transliterated) Sinhala text degrades real-world task performance compared to native Unicode Sinhala.

**Tasks:**
1. **Sinhala MMLU** — 4-way Multiple Choice (1,851 items)
2. **SOLD** — Binary Offensive Language Detection (2,500 items)
3. **Global PIQA** — 2-way Physical Intuition QA (100 items)

**Script Conditions:** `unicode` (native Sinhala) vs `romanized` (phonetic transliteration)

**Methodology:**
- **Zero-shot prompting** using standard instruct templates.
- **Native Chat Templates:** Uses `tokenizer.apply_chat_template()` to ensure instruct/chat models operate in their native assistant mode across diverse architectures (Phi, Llama, Mistral, Qwen, StableLM).
- **Deterministic decoding:** Greedy generation (`temperature=0.0`).
- **Robust Label Extraction:** Regex parsing with boundary safeguards against explanation tokens and language artifacts.

---
⚙️ **To switch models:** Change only `MODEL_ID` and `MODEL_LABEL` in the **Configuration** cell below.

## 1. Configuration
> **Change this cell to switch models.** Everything else runs automatically.

In [1]:
# ============================================================
# MODEL CONFIGURATION — CHANGE ONLY THIS CELL TO SWITCH MODELS
# ============================================================

MODEL_ID    = "Qwen/Qwen1.5-1.8B-Chat"           # HuggingFace model ID
MODEL_LABEL = "Qwen1.5-1.8B-Chat"                     # Short label for filenames & tables
MAX_NEW_TOKENS = 20                        # Max tokens to generate per response
TEMPERATURE    = 0.0                       # Greedy decoding (deterministic zero-shot)

# Dataset paths (update if your Kaggle dataset mount differs)
DATA_DIR = "/kaggle/input/datasets/sehandissanayake/sinhala-eval-datasets"  # Folder containing the 3 JSONL files

# Output prefix (auto-generated from MODEL_LABEL)
import re
OUTPUT_PREFIX = re.sub(r'[^a-z0-9]+', '_', MODEL_LABEL.lower()).strip('_')

print(f"Model:  {MODEL_ID}")
print(f"Label:  {MODEL_LABEL}")
print(f"Output: {OUTPUT_PREFIX}_*.csv")

Model:  Qwen/Qwen1.5-1.8B-Chat
Label:  Qwen1.5-1.8B-Chat
Output: qwen1_5_1_8b_chat_*.csv


## 2. Environment Setup & Hardware Verification

In [2]:
import sys
import os

!{sys.executable} -m pip install --upgrade --quiet transformers accelerate huggingface_hub

import torch
import json
import csv
import math
import time
import re
import transformers
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer

# Hardware check
device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | transformers {transformers.__version__} | {device_count} GPU(s)")
for i in range(device_count):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"  cuda:{i} -> {name} ({mem:.1f} GB)")
assert device_count > 0, "No GPU detected. Enable GPU T4 x2 in Session options."

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
PyTorch 2.10.0+cu128 | transformers 5.15.0 | 2 GPU(s)
  cuda:0 -> Tesla T4 (14.6 GB)
  cuda:1 -> Tesla T4 (14.6 GB)


In [3]:
# HuggingFace authentication (needed for gated models like Phi-4, Llama, etc.)
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Authenticated via Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = ""   # Paste your token here if Kaggle Secrets is unavailable
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("Authenticated via explicit token.")
    else:
        print(f"No HF_TOKEN found ({e}). Public models will still work.")

No HF_TOKEN found (Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 131143564 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.). Public models will still work.


## 3. Load Model & Tokenizer

In [4]:
print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN if HF_TOKEN else None,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Inspect chat template
has_template = hasattr(tokenizer, 'chat_template') and tokenizer.chat_template is not None
print(f"Chat template available: {has_template}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN if HF_TOKEN else None,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
model.eval()
model.config.use_cache = True  # Enable KV cache for generation

print(f"Model loaded successfully.")
print(f"Precision: {model.dtype}")
if hasattr(model, 'hf_device_map'):
    print(f"Device map: {model.hf_device_map}")

Loading Qwen/Qwen1.5-1.8B-Chat...


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.67GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

Model loaded successfully.
Precision: torch.float16
Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


## 4. Load Evaluation Datasets

In [5]:
def load_jsonl(filepath):
    """Load a JSONL file into a list of dicts."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

# Load all three datasets
mmlu_data  = load_jsonl(os.path.join(DATA_DIR, "sinhala_mmlu.jsonl"))
sold_data  = load_jsonl(os.path.join(DATA_DIR, "sold.jsonl"))
piqa_data  = load_jsonl(os.path.join(DATA_DIR, "global_piqa.jsonl"))

print(f"Sinhala MMLU : {len(mmlu_data):,} items (4-way MCQ)")
print(f"SOLD         : {len(sold_data):,} items (binary classification)")
print(f"Global PIQA  : {len(piqa_data):,} items (2-way MCQ)")
print(f"\nTotal items  : {len(mmlu_data) + len(sold_data) + len(piqa_data):,}")

Sinhala MMLU : 1,851 items (4-way MCQ)
SOLD         : 2,500 items (binary classification)
Global PIQA  : 100 items (2-way MCQ)

Total items  : 4,451


## 5. Standardized Prompt Templates & Robust Label Extraction

Design decisions for research reproducibility:
1. **Clear, deterministic instruction** specifying the expected answer format.
2. **Strict Regex Boundary Matching**: Prevents false positive matching of English articles (`'a'`), romanized word endings (`'ya.'`), or internal substrings (`'offensive'`).
3. **LaTeX / Markdown Stripping**: Handles common model wrappers like `\text{A}`, `**B**`, or `Answer: C` gracefully.

In [6]:
# ============================================================
# PROMPT TEMPLATES
# ============================================================

def build_mcq_prompt(question, options, option_labels=None):
    """
    Build a zero-shot MCQ prompt for Sinhala MMLU or Global PIQA.
    """
    if option_labels is None:
        option_labels = [chr(65 + i) for i in range(len(options))]  # A, B, C, D...

    options_str = "\n".join(f"{lbl}. {opt}" for lbl, opt in zip(option_labels, options))
    label_choices = ", ".join(option_labels)

    prompt = (
        f"Question: {question}\n\n"
        f"{options_str}\n\n"
        f"Answer with only the letter of the correct option ({label_choices}).\n"
        f"Answer:"
    )
    return prompt, option_labels


def build_sold_prompt(text):
    """
    Build a zero-shot binary classification prompt for offensive language detection.
    """
    prompt = (
        f"Classify the following Sinhala text as either 'NOT' (not offensive) or 'OFF' (offensive).\n\n"
        f"Text: {text}\n\n"
        f"Respond with only 'NOT' or 'OFF'.\n"
        f"Answer:"
    )
    return prompt


# ============================================================
# ROBUST LABEL EXTRACTION
# ============================================================

def extract_mcq_label(generated_text, valid_labels):
    """
    Extract the MCQ option letter (A, B, C, D) from generated text with strict boundary safeguards.
    """
    if not generated_text:
        return "INVALID"
    text = generated_text.strip()

    pattern_labels = "|".join(re.escape(lbl) for lbl in valid_labels)

    # 1. Match leading answer: 'A', 'A.', '(A)', '**A**', '\text{A}', 'Answer: A'
    prefix_match = re.match(
        rf'^(?:(?:the\s+)?(?:correct\s+)?(?:answer|option|choice)\s*(?:is|=|:)?\s*)?(?:\\text\{{|\\textbf\{{)?\s*[\*\(#\[\s]*({pattern_labels})[\*\)#\]\.\s:\n]',
        text,
        re.IGNORECASE
    )
    if prefix_match:
        return prefix_match.group(1).upper()

    # Exact single-letter / standalone letter at start
    if text and text[0].upper() in valid_labels:
        if len(text) == 1 or not text[1].isalnum():
            return text[0].upper()

    # 2. Look for explicit declaration in the body: "The answer is (A)", "Answer: B", "option C"
    body_match = re.search(
        rf'(?:(?:the\s+)?(?:correct\s+)?(?:answer|option|choice)\s*(?:is|=|:)?\s*)(?:\\text\{{)?\s*[\*\(#\[\s]*({pattern_labels})\b',
        text,
        re.IGNORECASE
    )
    if body_match:
        return body_match.group(1).upper()

    # 3. Look for standalone uppercase letter with delimiters (e.g. " A. ", " (B) ", " C: ")
    standalone_match = re.search(rf'(?:^|\s|\()({pattern_labels})(?:\.|\)|\:|\s|$)', text)
    if standalone_match:
        return standalone_match.group(1).upper()

    return "INVALID"


def extract_sold_label(generated_text):
    """
    Extract NOT or OFF with whole-word boundary checks.
    """
    if not generated_text:
        return "INVALID"
    text = generated_text.strip()

    # 1. Match at start: "NOT", "OFF", "**NOT**", "Answer: NOT"
    start_match = re.match(
        r'^(?:(?:the\s+)?(?:label|answer|classification)\s*(?:is|=|:)?\s*)?[\*\(#\[\s]*(NOT|OFF)\b',
        text,
        re.IGNORECASE
    )
    if start_match:
        return start_match.group(1).upper()

    # 2. Search for explicit declaration in body: "answer is NOT"
    body_match = re.search(
        r'(?:(?:the\s+)?(?:label|answer|classification)\s*(?:is|=|:)?\s*)[\*\(#\[\s]*(NOT|OFF)\b',
        text,
        re.IGNORECASE
    )
    if body_match:
        return body_match.group(1).upper()

    # 3. Search for standalone whole-word NOT or OFF (avoids matching inside "offensive")
    matches = list(re.finditer(r'\b(NOT|OFF)\b', text, re.IGNORECASE))
    if matches:
        return matches[0].group(1).upper()

    return "INVALID"


print("Standardized prompt templates and robust extraction functions defined.")

# Verification test
p_mcq, lbls = build_mcq_prompt("Sample question?", ["opt1", "opt2", "opt3", "opt4"])
print(f"\n--- Sample MCQ Prompt ---\n{p_mcq}")
print(f"\n--- Sample SOLD Prompt ---\n{build_sold_prompt('Sample text')}")

Standardized prompt templates and robust extraction functions defined.

--- Sample MCQ Prompt ---
Question: Sample question?

A. opt1
B. opt2
C. opt3
D. opt4

Answer with only the letter of the correct option (A, B, C, D).
Answer:

--- Sample SOLD Prompt ---
Classify the following Sinhala text as either 'NOT' (not offensive) or 'OFF' (offensive).

Text: Sample text

Respond with only 'NOT' or 'OFF'.
Answer:


## 6. Generation Engine with Native Chat Template

Instruct/chat models are trained on formatted conversation turns. Wrapping prompts with `tokenizer.apply_chat_template()` ensures the model operates in its native instruction-following mode, minimizing hallucinations, formatting errors, and unsolicited explanations.

In [7]:
@torch.inference_mode()
def generate_response(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    """
    Generate a response from the instruct model given a prompt string.
    Automatically applies tokenizer's chat template if available to standardize
    instruction-following behavior across model families.
    """
    if hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        formatted_prompt = prompt

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048  # Reserve space for generation
    ).to(device)

    input_len = inputs["input_ids"].shape[1]

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0,
        "pad_token_id": tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
    }
    if temperature > 0:
        gen_kwargs["temperature"] = temperature

    outputs = model.generate(**inputs, **gen_kwargs)

    # Decode only the NEW tokens (exclude the prompt)
    generated_ids = outputs[0][input_len:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return generated_text, input_len, len(generated_ids)

print("Generation engine initialized.")

Generation engine initialized.


## 7. Pilot Run

Sanity check on 5 items from each task × each script condition before the full evaluation.

In [8]:
print("=" * 75)
print("PILOT RUN — 5 items per task per script condition")
print("=" * 75)

# --- MMLU Pilot ---
print("\n--- Sinhala MMLU (MCQ) ---")
for item in mmlu_data[:5]:
    for script in ["unicode", "romanized"]:
        s = item[script]
        prompt, labels = build_mcq_prompt(s["text"], s["options"])
        gen_text, n_in, n_out = generate_response(prompt)
        pred = extract_mcq_label(gen_text, labels)
        correct = "✓" if pred == item["label"] else "✗"
        print(f"  [{script:9}] id={item['id']} | gold={item['label']} pred={pred} {correct} | raw='{repr(gen_text[:60])}'")

# --- SOLD Pilot ---
print("\n--- SOLD (Binary) ---")
for item in sold_data[:5]:
    for script in ["unicode", "romanized"]:
        s = item[script]
        prompt = build_sold_prompt(s["text"])
        gen_text, n_in, n_out = generate_response(prompt)
        pred = extract_sold_label(gen_text)
        correct = "✓" if pred == item["label"] else "✗"
        print(f"  [{script:9}] id={item['id']} | gold={item['label']} pred={pred} {correct} | raw='{repr(gen_text[:60])}'")

# --- PIQA Pilot ---
print("\n--- Global PIQA (MCQ) ---")
for item in piqa_data[:5]:
    for script in ["unicode", "romanized"]:
        s = item[script]
        prompt, labels = build_mcq_prompt(s["text"], s["options"])
        gen_text, n_in, n_out = generate_response(prompt)
        pred = extract_mcq_label(gen_text, labels)
        correct = "✓" if pred == item["label"] else "✗"
        print(f"  [{script:9}] id={item['id']} | gold={item['label']} pred={pred} {correct} | raw='{repr(gen_text[:60])}'")

print("\n" + "=" * 75)
print("Pilot complete. Check output above for clean extractions.")
print("=" * 75)

PILOT RUN — 5 items per task per script condition

--- Sinhala MMLU (MCQ) ---
  [unicode  ] id=mmlu_0001 | gold=A pred=A ✓ | raw=''A. Aamsa Sri Lanka''
  [romanized] id=mmlu_0001 | gold=A pred=A ✓ | raw=''A. aamisa puujaa saha prathipaththi puujaa''
  [unicode  ] id=mmlu_0002 | gold=C pred=A ✗ | raw=''A.''
  [romanized] id=mmlu_0002 | gold=C pred=B ✗ | raw=''B. raamasaan uthsawaya''
  [unicode  ] id=mmlu_0003 | gold=A pred=A ✓ | raw=''A.''
  [romanized] id=mmlu_0003 | gold=A pred=A ✓ | raw=''A.''
  [unicode  ] id=mmlu_0004 | gold=B pred=A ✗ | raw=''A.''
  [romanized] id=mmlu_0004 | gold=B pred=A ✗ | raw=''A. nabinaayakathumaagee upathayi.''
  [unicode  ] id=mmlu_0005 | gold=C pred=A ✗ | raw=''A. 2''
  [romanized] id=mmlu_0005 | gold=C pred=A ✗ | raw=''The correct answer is A. 2. The sentence "bauddha sthuupa pr''

--- SOLD (Binary) ---
  [unicode  ] id=sold_0001 | gold=NOT pred=NOT ✓ | raw=''NOT''
  [romanized] id=sold_0001 | gold=NOT pred=NOT ✓ | raw=''NOT''
  [unicode  ] id=sold_0002

## 8. Full Evaluation — All Tasks

Run all three tasks across both script conditions. Results are streamed to CSV files incrementally (crash-safe).

In [9]:
def run_mcq_evaluation(dataset, dataset_name, output_csv):
    """
    Run MCQ evaluation (MMLU or PIQA) on both script conditions.
    Writes results incrementally to CSV.
    """
    print(f"\n{'='*65}")
    print(f"Evaluating: {dataset_name} ({len(dataset)} items)")
    print(f"Output:     {output_csv}")
    print(f"{'='*65}")

    with open(output_csv, mode='w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow([
            'id', 'gold_label',
            'unicode_pred', 'unicode_correct', 'unicode_raw_output',
            'unicode_n_input_tokens', 'unicode_n_output_tokens',
            'romanized_pred', 'romanized_correct', 'romanized_raw_output',
            'romanized_n_input_tokens', 'romanized_n_output_tokens'
        ])

        t0 = time.time()
        for i, item in enumerate(tqdm(dataset, desc=dataset_name)):
            row = [item['id'], item['label']]

            for script in ['unicode', 'romanized']:
                s = item[script]
                prompt, labels = build_mcq_prompt(s['text'], s['options'])
                gen_text, n_in, n_out = generate_response(prompt)
                pred = extract_mcq_label(gen_text, labels)
                correct = 1 if pred == item['label'] else 0

                row.extend([pred, correct, gen_text.strip()[:200], n_in, n_out])

            writer.writerow(row)

            # Flush every 50 items to disk
            if (i + 1) % 50 == 0:
                f.flush()

        elapsed = time.time() - t0

    print(f"Done in {elapsed/60:.1f} min ({elapsed/len(dataset):.2f} s/item)")
    return output_csv


def run_sold_evaluation(dataset, output_csv):
    """
    Run SOLD binary classification on both script conditions.
    Writes results incrementally to CSV.
    """
    print(f"\n{'='*65}")
    print(f"Evaluating: SOLD ({len(dataset)} items)")
    print(f"Output:     {output_csv}")
    print(f"{'='*65}")

    with open(output_csv, mode='w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow([
            'id', 'gold_label',
            'unicode_pred', 'unicode_correct', 'unicode_raw_output',
            'unicode_n_input_tokens', 'unicode_n_output_tokens',
            'romanized_pred', 'romanized_correct', 'romanized_raw_output',
            'romanized_n_input_tokens', 'romanized_n_output_tokens'
        ])

        t0 = time.time()
        for i, item in enumerate(tqdm(dataset, desc="SOLD")):
            row = [item['id'], item['label']]

            for script in ['unicode', 'romanized']:
                s = item[script]
                prompt = build_sold_prompt(s['text'])
                gen_text, n_in, n_out = generate_response(prompt)
                pred = extract_sold_label(gen_text)
                correct = 1 if pred == item['label'] else 0

                row.extend([pred, correct, gen_text.strip()[:200], n_in, n_out])

            writer.writerow(row)

            if (i + 1) % 50 == 0:
                f.flush()

        elapsed = time.time() - t0

    print(f"Done in {elapsed/60:.1f} min ({elapsed/len(dataset):.2f} s/item)")
    return output_csv

print("Evaluation runners ready.")

Evaluation runners ready.


In [10]:
# ============================================================
# RUN ALL THREE EVALUATIONS
# ============================================================

mmlu_csv = run_mcq_evaluation(
    mmlu_data, "Sinhala MMLU",
    f"{OUTPUT_PREFIX}_sinhala_mmlu.csv"
)

sold_csv = run_sold_evaluation(
    sold_data,
    f"{OUTPUT_PREFIX}_sold.csv"
)

piqa_csv = run_mcq_evaluation(
    piqa_data, "Global PIQA",
    f"{OUTPUT_PREFIX}_global_piqa.csv"
)

print("\n" + "="*65)
print("ALL EVALUATIONS COMPLETE")
print("="*65)


Evaluating: Sinhala MMLU (1851 items)
Output:     qwen1_5_1_8b_chat_sinhala_mmlu.csv


Sinhala MMLU: 100%|██████████| 1851/1851 [21:34<00:00,  1.43it/s]


Done in 21.6 min (0.70 s/item)

Evaluating: SOLD (2500 items)
Output:     qwen1_5_1_8b_chat_sold.csv


SOLD: 100%|██████████| 2500/2500 [06:31<00:00,  6.39it/s]


Done in 6.5 min (0.16 s/item)

Evaluating: Global PIQA (100 items)
Output:     qwen1_5_1_8b_chat_global_piqa.csv


Global PIQA: 100%|██████████| 100/100 [01:06<00:00,  1.51it/s]

Done in 1.1 min (0.66 s/item)

ALL EVALUATIONS COMPLETE


## 9. Results Analysis & Summary Table

In [11]:
import pandas as pd
import numpy as np

def analyze_results(csv_path, task_name, model_label=MODEL_LABEL):
    """
    Analyze evaluation results from CSV and compute accuracy metrics.
    Returns a summary dict.
    """
    df = pd.read_csv(csv_path)
    n = len(df)

    # Overall accuracy
    unicode_acc   = df['unicode_correct'].mean() * 100
    romanized_acc = df['romanized_correct'].mean() * 100

    # Invalid prediction rates
    unicode_invalid   = (df['unicode_pred'] == 'INVALID').mean() * 100
    romanized_invalid = (df['romanized_pred'] == 'INVALID').mean() * 100

    # Performance gap
    gap = unicode_acc - romanized_acc
    degradation = (gap / unicode_acc * 100) if unicode_acc > 0 else float('nan')

    print(f"\n{'='*70}")
    print(f"  {model_label} — {task_name} ({n:,} items)")
    print(f"{'='*70}")
    print(f"  {'Script':14} {'Accuracy':>10} {'Invalid%':>10}")
    print(f"  {'-'*38}")
    print(f"  {'Unicode':14} {unicode_acc:>9.2f}% {unicode_invalid:>9.2f}%")
    print(f"  {'Romanized':14} {romanized_acc:>9.2f}% {romanized_invalid:>9.2f}%")
    print(f"  {'-'*38}")
    print(f"  Gap (U - R):   {gap:>+9.2f} pp")
    print(f"  Degradation:   {degradation:>9.1f}%")
    print(f"{'='*70}")

    return {
        'Task': task_name,
        'N': n,
        'Unicode Acc (%)': round(unicode_acc, 2),
        'Romanized Acc (%)': round(romanized_acc, 2),
        'Gap (pp)': round(gap, 2),
        'Degradation (%)': round(degradation, 1),
        'Unicode Invalid (%)': round(unicode_invalid, 2),
        'Romanized Invalid (%)': round(romanized_invalid, 2),
    }


# Run analysis on all three tasks
results = []
results.append(analyze_results(mmlu_csv, "Sinhala MMLU"))
results.append(analyze_results(sold_csv, "SOLD"))
results.append(analyze_results(piqa_csv, "Global PIQA"))

# Combined summary table
summary_df = pd.DataFrame(results)
print(f"\n\n{'='*80}")
print(f"  {MODEL_LABEL} — EXTRINSIC EVALUATION SUMMARY")
print(f"{'='*80}")
display(summary_df.style.format({
    'Unicode Acc (%)': '{:.2f}',
    'Romanized Acc (%)': '{:.2f}',
    'Gap (pp)': '{:+.2f}',
    'Degradation (%)': '{:.1f}',
}).set_caption(f"{MODEL_LABEL} — Extrinsic Downstream Task Results").background_gradient(
    subset=['Gap (pp)'], cmap='RdYlGn'
))


  Qwen1.5-1.8B-Chat — Sinhala MMLU (1,851 items)
  Script           Accuracy   Invalid%
  --------------------------------------
  Unicode            25.82%      0.05%
  Romanized          25.45%      1.24%
  --------------------------------------
  Gap (U - R):       +0.38 pp
  Degradation:         1.5%

  Qwen1.5-1.8B-Chat — SOLD (2,500 items)
  Script           Accuracy   Invalid%
  --------------------------------------
  Unicode            59.40%      0.00%
  Romanized          59.40%      0.00%
  --------------------------------------
  Gap (U - R):       +0.00 pp
  Degradation:         0.0%

  Qwen1.5-1.8B-Chat — Global PIQA (100 items)
  Script           Accuracy   Invalid%
  --------------------------------------
  Unicode            49.00%      0.00%
  Romanized          52.00%      0.00%
  --------------------------------------
  Gap (U - R):       -3.00 pp
  Degradation:        -6.1%


  Qwen1.5-1.8B-Chat — EXTRINSIC EVALUATION SUMMARY


,Task,N,Unicode Acc (%),Romanized Acc (%),Gap (pp),Degradation (%),Unicode Invalid (%),Romanized Invalid (%)
0,Sinhala MMLU,1851,25.82,25.45,+0.38,1.5,0.050000,1.240000
1,SOLD,2500,59.40,59.40,+0.00,0.0,0.000000,0.000000
2,Global PIQA,100,49.00,52.00,-3.00,-6.1,0.000000,0.000000


## 10. Detailed Per-Strata Breakdown

Analyze performance across specific strata:
- **Sinhala MMLU** by subject domain (`Humanities`, `Language`, `Social_Science`, `Stem`)
- **SOLD** by ground truth label (`NOT`, `OFF`)
- **Global PIQA** by cultural specificity (`culturally_specific=True/False`)

In [12]:
def strata_breakdown(data, csv_path, task_name, strata_key):
    """
    Compute accuracy breakdown by strata (e.g., domain, label, culturally_specific).
    """
    df = pd.read_csv(csv_path)

    # Build strata mapping from original data
    strata_map = {item['id']: item['strata'].get(strata_key, 'unknown') for item in data}
    df['stratum'] = df['id'].map(strata_map)

    print(f"\n{'='*70}")
    print(f"  {MODEL_LABEL} — {task_name} — Breakdown by '{strata_key}'")
    print(f"{'='*70}")

    rows = []
    for stratum, group in df.groupby('stratum'):
        n = len(group)
        u_acc = group['unicode_correct'].mean() * 100
        r_acc = group['romanized_correct'].mean() * 100
        gap = u_acc - r_acc
        rows.append({
            strata_key: stratum,
            'N': n,
            'Unicode Acc (%)': round(u_acc, 2),
            'Romanized Acc (%)': round(r_acc, 2),
            'Gap (pp)': round(gap, 2)
        })

    breakdown_df = pd.DataFrame(rows)
    display(breakdown_df.style.format({
        'Unicode Acc (%)': '{:.2f}',
        'Romanized Acc (%)': '{:.2f}',
        'Gap (pp)': '{:+.2f}',
    }).background_gradient(
        subset=['Gap (pp)'], cmap='RdYlGn'
    ).set_caption(f"{task_name} — Strata: {strata_key}"))

    return breakdown_df


# MMLU by domain
mmlu_strata = strata_breakdown(mmlu_data, mmlu_csv, "Sinhala MMLU", "domain")

# SOLD by label
sold_strata = strata_breakdown(sold_data, sold_csv, "SOLD", "label")

# PIQA by cultural specificity
piqa_strata = strata_breakdown(piqa_data, piqa_csv, "Global PIQA", "culturally_specific")


  Qwen1.5-1.8B-Chat — Sinhala MMLU — Breakdown by 'domain'


,domain,N,Unicode Acc (%),Romanized Acc (%),Gap (pp)
0,Humanities,1162,26.94,26.76,+0.17
1,Language,155,25.16,25.16,+0.00
2,Social_Science,370,25.95,24.32,+1.62
3,Stem,164,18.29,18.90,-0.61



  Qwen1.5-1.8B-Chat — SOLD — Breakdown by 'label'


,label,N,Unicode Acc (%),Romanized Acc (%),Gap (pp)
0,NOT,1485,100.00,100.00,+0.00
1,OFF,1015,0.00,0.00,+0.00



  Qwen1.5-1.8B-Chat — Global PIQA — Breakdown by 'culturally_specific'


,culturally_specific,N,Unicode Acc (%),Romanized Acc (%),Gap (pp)
0,False,23,30.43,34.78,-4.35
1,True,77,54.55,57.14,-2.60


## 11. Save Combined Summary & File Verification

In [13]:
# Save the combined summary to a single CSV
summary_csv = f"{OUTPUT_PREFIX}_extrinsic_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"Summary saved to: {summary_csv}")

# Print all output files
print(f"\nAll output files:")
for f in [mmlu_csv, sold_csv, piqa_csv, summary_csv]:
    size = os.path.getsize(f) / 1024
    print(f"  {f} ({size:.1f} KB)")

print(f"\n✅ {MODEL_LABEL} extrinsic evaluation complete!")
print(f"\nTo evaluate another model:")
print(f"  1. Change MODEL_ID and MODEL_LABEL in Cell 1 (Configuration)")
print(f"  2. Restart runtime and re-run all cells")

Summary saved to: qwen1_5_1_8b_chat_extrinsic_summary.csv

All output files:
  qwen1_5_1_8b_chat_sinhala_mmlu.csv (151.3 KB)
  qwen1_5_1_8b_chat_sold.csv (113.7 KB)
  qwen1_5_1_8b_chat_global_piqa.csv (7.7 KB)
  qwen1_5_1_8b_chat_extrinsic_summary.csv (0.2 KB)

✅ Qwen1.5-1.8B-Chat extrinsic evaluation complete!

To evaluate another model:
  1. Change MODEL_ID and MODEL_LABEL in Cell 1 (Configuration)
  2. Restart runtime and re-run all cells
